# 33장 · LangChain 검색 도구와 ToolMessage

검색 함수를 도구로 등록하면 모델이 정해진 인자 형식으로 검색을 요청할 수 있습니다. ToolMessage에는 어느 요청의 결과인지 나타내는 ID가 들어갑니다.

검색 함수를 도구로 등록하고 AIMessage의 호출 요청을 실제 검색과 연결합니다. 반환한 ToolMessage 안에서 도구 호출 ID와 검색된 문서의 ID를 구분합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain-core 1.6.12, pydantic 2.14.1입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 도구 인자 스키마와 호출 요청

같은 retrieve를 수업용 검색 도구로 감쌉니다. 이름과 인자 규칙은 모델이 검색 요청을 작성할 때 읽습니다. 이 노트북에서는 모델이 보낼 법한 요청 한 건을 직접 만들어 네트워크 없이 실제 LangChain 도구에 전달합니다.


### 셀 01 · 도구 인자 스키마와 @tool

자료와 검색 함수는 30장과 같습니다. 맨 아래 search_documents의 이름과 k 인자 규칙만 찾아보세요.


In [1]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]

from langchain_core.tools import tool
from langchain_core.messages import AIMessage
from pydantic import BaseModel, ConfigDict, StrictInt, StrictStr, Field

class SearchArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    question: StrictStr
    k: StrictInt = Field(default=2, ge=1)

@tool("search_documents", args_schema=SearchArgs, response_format="content_and_artifact")
def search_documents(question: str, k: int = 2):
    """가상 도서관의 대출·휴관·반납 안내를 찾습니다."""
    if not question.strip():
        raise ValueError("질문을 입력하세요.")
    found = retrieve(question, k)
    return "\n".join(row["text"] for row in found), found

print(search_documents.name)

search_documents


출력된 search_documents는 등록한 도구의 이름입니다. 검색은 아직 실행되지 않았습니다.

도구에 질문을 건네야 검색 결과가 생깁니다. 다음 셀에서 AIMessage에 미리 넣은 인자를 읽고, 실제 도구를 호출한 뒤에야 원문과 출처를 확인할 수 있습니다.


### 도구 이름·설명·인자 스키마

<p>도구 명세에는 이름, 사용할 상황을 알려 주는 설명, 받을 인자의 구조가 들어갑니다. 이름 search_documents는 실행할 함수를 찾는 기준입니다. 설명은 가상 도서관의 대출·휴관·반납 자료를 검색한다는 범위를 알립니다. 짧아도 자료의 범위와 반환 목적이 드러나야 합니다.</p>

<p>question은 무엇을 찾을지, k는 문서를 최대 몇 개 받을지를 나타냅니다. 기본값 2는 생략할 수 있다는 뜻이며 반드시 두 문서가 나온다는 뜻은 아닙니다. Tool의 설명이 있어도 실제 모델이 항상 필요한 검색을 선택한다고 보장할 수는 없습니다.</p>

<table><caption>입력 규칙을 예상해 보기</caption><thead><tr><th scope="col">입력</th><th scope="col">처리</th><th scope="col">이유</th></tr>
</thead><tbody><tr><td>question="대출 기간은?", k=1</td><td>검색 실행</td><td>문자열 질문과 양의 정수</td></tr>
<tr><td>question=" "</td><td>검색 전 거부</td><td>공백뿐인 질문</td></tr>
<tr><td>k=True 또는 k="2"</td><td>검색 전 거부</td><td>엄격한 정수 규칙</td></tr>
<tr><td>k=0 또는 extra=1</td><td>검색 전 거부</td><td>양수 조건 또는 모르는 인자</td></tr>
</tbody></table>

<p>BaseModel·Field·StrictInt는 이 장에서 제공하는 검사 코드입니다. 이름의 뜻과 입력 결과만 읽고 클래스 내부를 새로 구현할 필요는 없습니다. Python에서는 True가 정수처럼 쓰일 수 있으므로 엄격한 검사를 두어 정수가 필요한 자리에 참·거짓 값이 들어오는 것을 막습니다. 추가 인자를 거부하면 오타를 조용히 무시하지 않습니다.</p>


### content_and_artifact의 두 반환값

<p>도구 함수는 검색 문장들을 줄바꿈으로 이은 문자열과 원래 레코드 리스트를 쌍으로 돌려줍니다. 첫 값은 읽을 내용, 둘째 값은 id와 page까지 갖춘 구조화된 근거입니다.</p>


### 셀 02 · 도구 호출 메시지 구성

질문과 k가 담긴 도구 호출을 AIMessage에서 꺼내세요.


In [2]:
request = AIMessage(content='', tool_calls=[{'id': 'search-1', 'name': 'search_documents', 'args': {'question': '대출 기간은?', 'k': 1}, 'type': 'tool_call'}])
call = request.tool_calls[0]
print(call['args'])

{'question': '대출 기간은?', 'k': 1}


대출 질문과 검색 상한 1이 나옵니다. 이 시점에는 문서가 없습니다.


## tool_call_id로 ToolMessage 연결

invoke에 인자만 넘길 때와 달리 호출 ID까지 포함한 요청을 건네면 ToolMessage에서 어느 요청의 결과인지 읽을 수 있습니다. 화면에는 실제 검색 자료를 싣고 메시지에는 다음 판단에 필요한 내용을 담습니다.


### 과제 03 · ToolMessage와 요청 ID

call을 도구로 실행한 뒤 tool_call_id와 첫 문서의 id를 표시하세요.

content에서 읽을 수 있는 값은 검색한 안내 문장입니다. id, source, page가 필요하면 같은 응답의 artifact를 읽으세요.


In [3]:
# call 전체를 search_documents.invoke에 넘기세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>reply = search_documents.invoke(call)
print(reply.tool_call_id, reply.artifact[0][&#x27;id&#x27;])</code></pre>
<p>실행 결과</p><pre><code>search-1 library-1</code></pre>
<p>search-1은 요청 식별자, library-1은 근거 문서 식별자입니다. 이 둘을 바꿔 쓰면 다음 판단에서 어떤 요청에 답이 왔는지 찾기 어렵습니다.</p>
</details>


### 요청 하나에 붙는 ID와 문서 여러 건에 붙는 ID

<p>search-1은 요청을 보낼 때 정한 값입니다. 검색 상한을 2로 바꿔 문서가 두 건 돌아와도 요청 ID는 하나입니다. 두 문서의 id는 artifact의 서로 다른 행에서 읽습니다.</p>


### 빈 artifact 처리

<p>주차 요금을 찾으면 요청 ID가 붙은 ToolMessage 안에서 artifact가 비어 있습니다. 화면 코드가 문서를 표시하기 전에 빈 목록을 확인할 자리입니다.</p>


### 잘못된 인자가 검색에 도달하기 전

<p>SearchArgs에서 k는 1 이상의 StrictInt입니다. 문자열 "1"이나 정수 0을 k로 넣으면 인자 검증에서 멈춥니다. 자료가 없어 빈 목록을 반환한 정상 검색과 발생 지점이 다릅니다.</p>


### 과제 04 · 도구가 정상 실행됐지만 자료가 없는 요청

새 호출 ID를 search-parking으로 정하고 주차 요금을 물으세요. 도구 응답에서 요청 ID와 artifact를 출력하세요. 빈 목록에서 첫 행을 꺼내는 코드가 왜 실패하는지도 확인하세요.


In [ ]:
parking_call = {**call, 'id': 'search-parking', 'args': {'question': '주차 요금은?', 'k': 1}}
# 호출 전체를 넘기고 응답을 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>parking_call = {**call, &#x27;id&#x27;: &#x27;search-parking&#x27;, &#x27;args&#x27;: {&#x27;question&#x27;: &#x27;주차 요금은?&#x27;, &#x27;k&#x27;: 1}}
reply = search_documents.invoke(parking_call)
print(reply.tool_call_id, reply.artifact)</code></pre>
<p>실행 결과</p><pre><code>search-parking []</code></pre>
<p>search-parking을 돌려받았으므로 해당 요청의 도구 실행은 끝났습니다. 빈 artifact에서 [0]을 읽으면 IndexError가 납니다. 먼저 목록이 비었는지 확인하면 자료 없음 안내를 표시하면서도 요청 기록을 지킬 수 있습니다.</p>
</details>


## 돌아보기

도구 응답이 빈 목록이면 호출 자체는 끝난 상태입니다. 검색 호출을 건너뛴 경우와 구별할 수 있도록 요청 ID, 실행 여부, 자료 목록을 함께 읽으세요.
